# Comparative Study of Deep CNN Architectures Using Transfer Learning — CIFAR-10

**CS3807 · Deep Learning Laboratory · Experiment 4**
Shiv Nadar University Chennai · B.Tech AI & Data Science · Semester V · AY 2026–27

---

### What's in this notebook

This notebook implements the full prescribed laboratory experiment: transfer learning with a
pretrained CNN on CIFAR-10, fine-tuning, a hyperparameter ablation study, and a from-scratch
comparison of five landmark CNN architectures (**LeNet-5, AlexNet, VGG16, GoogLeNet, ResNet50**).

| | |
|---|---|
| **Dataset** | CIFAR-10 (50,000 train / 10,000 test, 32×32×3, 10 classes) |
| **Core experiment** | Transfer learning with **VGG16** (ImageNet weights) → fine-tuning |
| **Ablation study** | Learning rate, batch size, epochs, optimizer, dense units, frozen layers |
| **Architecture comparison** | LeNet-5, AlexNet, VGG16, GoogLeNet (Inception-v1), ResNet50 — canonical `torchvision` implementations, trained from scratch on CIFAR-10 under a common, documented training budget (not a fully isolated ablation — see Section 9) |
| **Evaluation** | Accuracy, Precision, Recall, F1, Confusion Matrix, Classification Report |

> **Note on reproducibility:** This notebook is written to run top-to-bottom on a Kaggle GPU
> instance (P100/T4). Cells that require an actual training run to produce a number or a curve
> are left with clearly marked **`TODO AFTER EXECUTION`** placeholders — no results below have
> been fabricated. Run the notebook, then fill these in from the real output.
>
> **Note on frameworks:** Sections 1–8 and 11 follow the manual's instruction to load CIFAR-10 and
> build the transfer-learning pipeline with TensorFlow/Keras. Section 9 (Architecture Comparison)
> uses PyTorch/`torchvision` instead, solely because `torchvision.models` ships true canonical
> AlexNet and GoogLeNet implementations (Keras has neither), which keeps every architecture in that
> section authentic rather than hand-approximated. This is explained in full where it happens.

**Table of Contents**
1. [Setup & Configuration](#setup)
2. [Theory: Evolution of CNN Architectures](#theory)
3. [Task 1 — Dataset Preparation](#task1)
4. [Task 2 — Transfer Learning Model](#task2)
5. [Task 3 — Baseline Training](#task3)
6. [Task 4 — Fine-Tuning](#task4)
7. [Task 5 — Model Evaluation](#task5)
8. [Hyperparameter Study](#hp)
9. [Architecture Comparison](#arch)
10. [Discussion Questions](#discussion)
11. [Additional Exercises](#exercises)
12. [Conclusion & References](#conclusion)


## 1. Setup & Configuration <a id="setup"></a>

All imports, the global random seed, and shared configuration live here so that every later
section reads from a single source of truth — this keeps the hyperparameter study and the
architecture comparison honest about what was held constant.

In [ ]:
# Core imports
import os, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------------------------------------------------
# Global configuration (single source of truth for the whole notebook)
# ---------------------------------------------------------------------
CONFIG = {
    "num_classes": 10,
    "img_shape_native": (32, 32, 3),      # CIFAR-10 native resolution
    "class_names": ["airplane", "automobile", "bird", "cat", "deer",
                     "dog", "frog", "horse", "ship", "truck"],

    # Task 3 — baseline training (as specified by the manual)
    "baseline_optimizer": "adam",
    "baseline_lr": 0.001,
    "baseline_batch_size": 32,
    "baseline_epochs": 15,                # manual range: 10-20
    "baseline_dense_units": 128,
    "baseline_frozen": "all",             # convolutional base fully frozen

    # Task 4 — fine-tuning
    "finetune_epochs": 8,                 # manual range: 5-10
    "finetune_lr": 1e-5,                  # smaller LR to avoid destroying pretrained features

    "figure_dpi": 600,
    "fig_dir": "figures",
}
os.makedirs(CONFIG["fig_dir"], exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print(json.dumps(CONFIG, indent=2))

In [ ]:
def save_fig(fig, name):
    '''Save a figure at publication-quality DPI and keep it visible inline.'''
    path = os.path.join(CONFIG["fig_dir"], f"{name}.png")
    fig.savefig(path, dpi=CONFIG["figure_dpi"], bbox_inches="tight")
    print(f"Saved: {path}")

def count_params(model):
    trainable = np.sum([np.prod(v.shape) for v in model.trainable_weights])
    non_trainable = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])
    return int(trainable), int(non_trainable), int(trainable + non_trainable)

def evaluate_on_test(model, x=None, y_labels=None, batch_size=128):
    '''Predict on the held-out CIFAR-10 test set with a Keras model and return accuracy.

    Defaults to the notebook's global `x_test` / `y_true` (set in Task 1 / Task 5), so it can be
    called simply as `evaluate_on_test(model)` anywhere after those exist -- e.g. for the ResNet50
    transfer-learning repeat in the Additional Exercises section.
    '''
    if x is None:
        x = x_test
    if y_labels is None:
        y_labels = y_true
    preds = np.argmax(model.predict(x, batch_size=batch_size, verbose=0), axis=1)
    return accuracy_score(y_labels, preds)

## 2. Theory: Evolution of CNN Architectures <a id="theory"></a>

### 2.1 Timeline

| Model | Year | Depth | Parameters | Major Contribution |
|---|---|---|---|---|
| LeNet-5 | 1998 | 7 | ~60K | First practical CNN |
| AlexNet | 2012 | 8 | ~61M | ReLU activation, Dropout, GPU training |
| VGG16 | 2014 | 16 | ~138M | Uniform 3×3 convolution filters |
| GoogLeNet | 2014 | 22 | ~6.8M | Inception modules for multi-scale features |
| ResNet50 | 2015 | 50 | ~25.6M | Residual learning via skip connections |

### 2.2 LeNet-5 (LeCun et al., 1998)
The first practical CNN, built for handwritten digit recognition. It stacks two
convolution + average-pooling stages followed by fully connected layers, and showed that a
network could learn useful image features directly from pixels instead of hand-crafted
descriptors. It is small, fast to train, and well suited to simple, low-resolution tasks such as
OCR — but it lacks the capacity for complex, high-resolution natural images.

### 2.3 AlexNet (Krizhevsky et al., 2012)
Won ILSVRC-2012 by a large margin. Its key innovations were the **ReLU** activation
(faster convergence than sigmoid/tanh), **Dropout** (regularization against overfitting in a
~61M-parameter network), and training across GPUs, which made deep networks practical at scale.

### 2.4 VGG16 (Simonyan & Zisserman, 2014)
Showed that depth, not filter size, was the key to accuracy: stacking multiple **3×3**
convolutions gives the same effective receptive field as a larger filter (e.g. two 3×3 layers ≈
one 5×5 layer) with fewer parameters and an extra non-linearity, at the cost of a very large
number of fully-connected parameters (~138M total).

### 2.5 GoogLeNet / Inception-v1 (Szegedy et al., 2014)
Introduced the **Inception module**: instead of choosing one filter size, it applies 1×1, 3×3,
5×5 convolutions and a max-pool branch *in parallel* and concatenates their outputs, capturing
multi-scale features in a single layer. 1×1 convolutions are used as "bottlenecks" before the
expensive 3×3/5×5 branches, which is why GoogLeNet reaches 22 layers with only ~6.8M parameters —
far fewer than VGG16.

**Inception module (schematic):**

```
                    ┌─────────────┐
Input ──┬──────────►│ 1x1 conv    │──┐
        │           └─────────────┘  │
        ├──►[1x1]──►│ 3x3 conv    │──┤
        │           └─────────────┘  ├──► Concatenate ──► Output
        ├──►[1x1]──►│ 5x5 conv    │──┤
        │           └─────────────┘  │
        └──►[3x3 max-pool]──►[1x1]───┘
```

### 2.6 ResNet (He et al., 2015)
Very deep plain networks are hard to optimize — accuracy saturates and then *degrades* with
depth (the vanishing/exploding gradient and degradation problem), not because of overfitting but
because deep stacks of non-linear layers struggle to learn even an identity mapping. ResNet
reframes each block to learn a **residual** function:

$$\mathcal{F}(x) = \mathcal{H}(x) - x \quad\Longrightarrow\quad \text{Output} = \mathcal{F}(x) + x$$

where $\mathcal{H}(x)$ is the desired underlying mapping, $x$ is the identity (skip connection),
and $\mathcal{F}(x)$ is what the stacked layers actually need to learn. If the identity is already
optimal, the network only needs to push $\mathcal{F}(x) \to 0$, which is far easier than learning
an identity mapping from scratch through several non-linear layers. This lets gradients flow
directly through the skip connections, enabling networks with 50+ layers to train reliably.

### 2.7 Dilated (Atrous) Convolution
A dilated convolution inserts gaps (spacing) between kernel elements, controlled by a
**dilation rate** $D$. $D=1$ is an ordinary convolution; $D>1$ enlarges the receptive field
**without** adding parameters or increasing computation:

$$
\text{3×3 kernel, } D=2 \;\rightarrow\;
\begin{bmatrix} 1 & 0 & 2 \\ 0 & 3 & 0 \\ 4 & 0 & 5 \end{bmatrix}
$$

This is heavily used in **semantic segmentation**, medical imaging, satellite imagery, and object
localization, where a large context window is needed but downsampling (which loses spatial
resolution) is undesirable.

### 2.8 Transpose Convolution
While pooling and strided convolution *reduce* spatial resolution, transpose convolution (a.k.a.
"deconvolution" or fractionally-strided convolution) performs a **learnable upsampling** — it
maps a small feature map to a larger one using trainable weights, unlike fixed upsampling
(nearest-neighbour/bilinear). It is the standard building block of decoders in autoencoders,
GAN generators, super-resolution networks, and segmentation decoders.

### 2.9 Transfer Learning & Fine-Tuning
A CNN trained on a large dataset (ImageNet, 1.2M images / 1000 classes) learns a hierarchy of
features — edges and textures in early layers, parts and objects in later layers — that transfer
well to other vision tasks. Transfer learning reuses that hierarchy instead of training from
scratch:

1. Load a pretrained model (convolutional base).
2. Remove the original ImageNet classifier head.
3. Freeze the convolutional base (its weights are not updated).
4. Add new, randomly-initialized Dense layers for the new task (CIFAR-10, 10 classes).
5. Train **only** the new head — the base acts as a fixed feature extractor.
6. **Fine-tune**: unfreeze the top (most task-specific) convolutional block and continue training
   at a much smaller learning rate, letting the network adapt its high-level features to the new
   domain without destroying the low-level features learned from ImageNet.

This is effective because early convolutional filters (edges, colour blobs, textures) are
largely task-agnostic, so only the top of the network needs to specialize — giving faster
convergence, better accuracy on small datasets, and lower compute cost than training from
scratch.


## 3. Task 1 — Dataset Preparation <a id="task1"></a>

Per the manual: load CIFAR-10 via `tf.keras.datasets`, normalize pixel values to $[0,1]$,
display sample images, and print dataset dimensions.

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

# One-hot encode labels (needed for categorical cross-entropy)
y_train = to_categorical(y_train_raw, CONFIG["num_classes"])
y_test = to_categorical(y_test_raw, CONFIG["num_classes"])

print("Training images :", x_train.shape, "| labels:", y_train.shape)
print("Testing images  :", x_test.shape, "| labels:", y_test.shape)
print("Pixel value range after normalization: [{:.3f}, {:.3f}]".format(x_train.min(), x_train.max()))

In [ ]:
# Plot 1 (mandatory): 10 sample CIFAR-10 images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(x_train), 10, replace=False)
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(x_train[idx])
    ax.set_title(CONFIG["class_names"][int(y_train_raw[idx])], fontsize=11)
    ax.axis("off")
fig.suptitle("Sample CIFAR-10 Images", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "01_sample_cifar10_images")
plt.show()

## 4. Task 2 — Transfer Learning Model <a id="task2"></a>

The manual allows a choice of **VGG16, ResNet50, MobileNetV2, or EfficientNetB0 (optional)**.

**Choice: VGG16.** It has a simple, uniform, well-documented architecture, is straightforward to
explain (no residual/inception branching to reason about when discussing the transfer-learning
workflow itself), and is a standard, reliable baseline for transfer learning on small images. Its
computational cost is reasonable for CIFAR-10 once the convolutional base is frozen. ResNet50 is
used later, both in the additional-exercises section (repeating the transfer-learning pipeline)
and in the architecture-comparison section.

The transfer-learning workflow (exactly as specified):
1. Load pretrained ImageNet weights.
2. Remove the original classification (top) layer.
3. Freeze the convolutional base.
4. Add a Global Average Pooling layer.
5. Add a Dense layer with ReLU activation.
6. Add the output layer with Softmax activation.

Since VGG16's ImageNet weights expect inputs of at least 32×32 (and were trained at 224×224), we
add a `Resizing` layer so the pretrained filters see images at the resolution they were trained
on — CIFAR-10 images are upsampled to 224×224 before entering the frozen base. This resizing is a
technical necessity for using ImageNet weights correctly, not a deviation from the manual's
workflow.

In [ ]:
def build_transfer_model(base_name="VGG16", dense_units=128, frozen="all",
                          input_shape=(32, 32, 3), target_size=(224, 224),
                          num_classes=10):
    '''
    Generic transfer-learning model builder used throughout the notebook
    (Task 2, the hyperparameter study, and Additional Exercise 2 / ResNet50 repeat).

    base_name : 'VGG16' or 'ResNet50'
    frozen    : 'all' (freeze entire conv base) or 'partial' (freeze all but last block)
    '''
    inputs = keras.Input(shape=input_shape)
    x = layers.Resizing(*target_size)(inputs)

    if base_name == "VGG16":
        preprocess = keras.applications.vgg16.preprocess_input
        base = VGG16(weights="imagenet", include_top=False, input_shape=(*target_size, 3))
        last_block_prefix = "block5"
    elif base_name == "ResNet50":
        preprocess = keras.applications.resnet50.preprocess_input
        base = ResNet50(weights="imagenet", include_top=False, input_shape=(*target_size, 3))
        last_block_prefix = "conv5"
    else:
        raise ValueError(f"Unsupported base_name: {base_name}")

    x = layers.Lambda(preprocess, name="preprocess")(x)

    if frozen == "all":
        base.trainable = False
    elif frozen == "partial":
        base.trainable = True
        for layer in base.layers:
            layer.trainable = last_block_prefix in layer.name
    else:
        raise ValueError("frozen must be 'all' or 'partial'")

    x = base(x, training=False if frozen == "all" else None)
    x = layers.GlobalAveragePooling2D(name="gap")(x)                    # Step 4
    x = layers.Dense(dense_units, activation="relu", name="fc1")(x)     # Step 5
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)  # Step 6

    model = keras.Model(inputs, outputs, name=f"{base_name}_transfer_{frozen}")
    return model, base


vgg_model, vgg_base = build_transfer_model(
    base_name="VGG16",
    dense_units=CONFIG["baseline_dense_units"],
    frozen=CONFIG["baseline_frozen"],
)
vgg_model.summary()

trainable_p, nontrainable_p, total_p = count_params(vgg_model)
print(f"\nTrainable params     : {trainable_p:,}")
print(f"Non-trainable params : {nontrainable_p:,}")
print(f"Total params         : {total_p:,}")

## 5. Task 3 — Baseline Training <a id="task3"></a>

Training configuration exactly as specified by the manual:

| Setting | Value |
|---|---|
| Optimizer | Adam |
| Learning rate | 0.001 |
| Batch size | 32 |
| Epochs | 15 (within the manual's 10–20 range) |
| Loss | Categorical Cross-Entropy |
| Metric | Accuracy |

A held-out 10% validation split from the training set is used to track validation accuracy/loss
per epoch (required for the mandatory validation curves), while the official CIFAR-10 test set is
reserved untouched for the Task 5 evaluation.

In [ ]:
vgg_model.compile(
    optimizer=optimizers.Adam(learning_rate=CONFIG["baseline_lr"]),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

t0 = time.time()
history_baseline = vgg_model.fit(
    x_train, y_train,
    validation_split=0.1,
    batch_size=CONFIG["baseline_batch_size"],
    epochs=CONFIG["baseline_epochs"],
    verbose=1,
)
baseline_training_time = time.time() - t0
print(f"\nBaseline training time: {baseline_training_time:.1f} s")

In [ ]:
# Plots 2-5 (mandatory), presented as a clean 2x2 subplot:
# Training Accuracy, Validation Accuracy, Training Loss, Validation Loss
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(history_baseline.history["accuracy"], color="#1f77b4")
axes[0, 0].set_title("Training Accuracy")
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Accuracy")

axes[0, 1].plot(history_baseline.history["val_accuracy"], color="#ff7f0e")
axes[0, 1].set_title("Validation Accuracy")
axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Accuracy")

axes[1, 0].plot(history_baseline.history["loss"], color="#1f77b4")
axes[1, 0].set_title("Training Loss")
axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("Loss")

axes[1, 1].plot(history_baseline.history["val_loss"], color="#ff7f0e")
axes[1, 1].set_title("Validation Loss")
axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("Loss")

for ax in axes.flat:
    ax.grid(alpha=0.3)
fig.suptitle("Baseline Training Curves (VGG16 Transfer Learning)", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "02_baseline_training_curves")
plt.show()

**TODO AFTER EXECUTION:** Write a 2–3 line data-driven inference from the four curves above
(e.g. whether the model is overfitting, underfitting, or converging well, and how many epochs it
took to plateau).

## 6. Task 4 — Fine-Tuning <a id="task4"></a>

1. Unfreeze the last convolutional block of VGG16 (`block5_*`).
2. Continue training for 8 more epochs (within the manual's 5–10 range) at a much smaller
   learning rate ($1\times10^{-5}$) so the pretrained weights adapt gently instead of being
   overwritten.
3. Compare accuracy before vs. after fine-tuning.

In [ ]:
# Unfreeze only the last convolutional block ("block5_*") of the VGG16 base
vgg_base.trainable = True
for layer in vgg_base.layers:
    layer.trainable = "block5" in layer.name

trainable_p_ft, nontrainable_p_ft, total_p_ft = count_params(vgg_model)
print(f"Trainable params after unfreezing block5: {trainable_p_ft:,} "
      f"(was {trainable_p:,} during baseline training)")

vgg_model.compile(
    optimizer=optimizers.Adam(learning_rate=CONFIG["finetune_lr"]),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

t0 = time.time()
history_finetune = vgg_model.fit(
    x_train, y_train,
    validation_split=0.1,
    batch_size=CONFIG["baseline_batch_size"],
    epochs=CONFIG["finetune_epochs"],
    verbose=1,
)
finetune_training_time = time.time() - t0
print(f"\nFine-tuning time: {finetune_training_time:.1f} s")
print(f"Total training time (baseline + fine-tune): "
      f"{baseline_training_time + finetune_training_time:.1f} s")

In [ ]:
# Before vs. after fine-tuning comparison
acc_before = history_baseline.history["val_accuracy"][-1]
acc_after = history_finetune.history["val_accuracy"][-1]

fig, ax = plt.subplots(figsize=(7, 5))
epochs_before = range(1, len(history_baseline.history["val_accuracy"]) + 1)
epochs_after = range(len(epochs_before) + 1,
                      len(epochs_before) + len(history_finetune.history["val_accuracy"]) + 1)

ax.plot(epochs_before, history_baseline.history["val_accuracy"], label="Frozen base (baseline)", color="#1f77b4")
ax.plot(epochs_after, history_finetune.history["val_accuracy"], label="Fine-tuned (block5 unfrozen)", color="#d62728")
ax.axvline(len(epochs_before) + 0.5, linestyle="--", color="gray", alpha=0.7, label="Fine-tuning starts")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
ax.set_title("Validation Accuracy: Before vs. After Fine-Tuning")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, "03_finetune_before_after")
plt.show()

print(f"Validation accuracy before fine-tuning: {acc_before:.4f}")
print(f"Validation accuracy after fine-tuning : {acc_after:.4f}")
print(f"Absolute change: {acc_after - acc_before:+.4f}")

**TODO AFTER EXECUTION:** Write a 2–3 line data-driven conclusion comparing accuracy before
vs. after fine-tuning (did fine-tuning help, hurt, or make little difference, and why might that
be, given the learning rate and how many layers were unfrozen).

## 7. Task 5 — Model Evaluation <a id="task5"></a>

Evaluate the fine-tuned model on the untouched CIFAR-10 test set: Accuracy, Precision, Recall,
F1-score, Confusion Matrix, and a full Classification Report.

In [ ]:
y_pred_probs = vgg_model.predict(x_test, batch_size=64, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test_raw.flatten()

test_accuracy = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
test_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
test_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Test Accuracy  : {test_accuracy:.4f}")
print(f"Test Precision : {test_precision:.4f}  (macro-averaged)")
print(f"Test Recall    : {test_recall:.4f}  (macro-averaged)")
print(f"Test F1-score  : {test_f1:.4f}  (macro-averaged)")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=CONFIG["class_names"]))

In [ ]:
# Plot 6 (mandatory): Confusion Matrix -- standalone figure
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CONFIG["class_names"], yticklabels=CONFIG["class_names"], ax=ax)
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_title("Confusion Matrix — VGG16 Transfer Learning (Test Set)", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "04_confusion_matrix")
plt.show()

In [ ]:
# Plot 7 (mandatory, optional-per-manual): Misclassified images
wrong_idx = np.where(y_pred != y_true)[0]
show_idx = rng.choice(wrong_idx, min(10, len(wrong_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, idx in zip(axes.flat, show_idx):
    ax.imshow(x_test[idx])
    true_lbl = CONFIG["class_names"][y_true[idx]]
    pred_lbl = CONFIG["class_names"][y_pred[idx]]
    ax.set_title(f"True: {true_lbl}\nPred: {pred_lbl}", fontsize=9, color="crimson")
    ax.axis("off")
fig.suptitle("Sample Misclassified Test Images", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "05_misclassified_images")
plt.show()

print(f"Total misclassified: {len(wrong_idx)} / {len(y_true)} "
      f"({100 * len(wrong_idx) / len(y_true):.2f}%)")

**TODO AFTER EXECUTION:**
- 2–3 line inference on the confusion matrix (which class pairs are most confused, and why that
  might make visual sense — e.g. cat/dog, automobile/truck).
- 2–3 line inference on the misclassified images (any common visual pattern: occlusion, unusual
  pose, low contrast, ambiguous class boundary, etc.).

In [ ]:
results_table = pd.DataFrame([{
    "Training Accuracy": history_finetune.history["accuracy"][-1],
    "Testing Accuracy": test_accuracy,
    "Precision": test_precision,
    "Recall": test_recall,
    "F1-score": test_f1,
    "Training Time (s)": round(baseline_training_time + finetune_training_time, 1),
    "Total Parameters": total_p,
}]).T.rename(columns={0: "Value"})
results_table

## 8. Hyperparameter Study <a id="hp"></a>

The manual specifies six hyperparameters and two values each. A full factorial grid would be
$2\times3\times2\times2\times2\times2 = 96$ training runs — not "computationally reasonable" for
a lab notebook. Instead we use a **one-factor-at-a-time (OFAT)** ablation: start from the
baseline configuration below, vary **one** hyperparameter across its specified values while
holding all others fixed, and record the result. This still investigates every single value
listed in the manual, while keeping the total run count linear instead of exponential.

To keep the *total* compute budget reasonable, every ablation run below uses:
- a **stratified 20% subset** of the training set (10,000 images) and the full test set for
  scoring,
- a fixed, shorter **6 epochs** per run (except the dedicated "Epochs" sweep, whose entire point
  is to vary epoch count).

This is a deliberate, clearly-stated trade-off between the manual's requirement to test every
value and the requirement to keep the notebook computationally reasonable — absolute accuracy
numbers in this section are therefore **not directly comparable** to the Task 3/4/5 numbers above,
which use the full dataset. Only the *relative* comparison within this section is meaningful.

**Baseline configuration (held constant unless the row varies it):**

| Hyperparameter | Baseline value |
|---|---|
| Learning rate | 0.001 |
| Batch size | 32 |
| Epochs | 6 (ablation-only; full runs use 10-20, see Task 3/4) |
| Optimizer | Adam |
| Dense units | 128 |
| Frozen layers | All |

In [ ]:
# Stratified subset for the ablation study (keeps class balance)
from sklearn.model_selection import train_test_split

HP_SUBSET_FRACTION = 0.20
HP_EPOCHS_DEFAULT = 6

_, x_hp, _, y_hp_raw = train_test_split(
    x_train, y_train_raw, test_size=HP_SUBSET_FRACTION,
    stratify=y_train_raw, random_state=SEED,
)
y_hp = to_categorical(y_hp_raw, CONFIG["num_classes"])
print("Hyperparameter-study subset:", x_hp.shape, y_hp.shape)

HP_BASELINE = dict(lr=0.001, batch_size=32, epochs=HP_EPOCHS_DEFAULT,
                    optimizer="adam", dense_units=128, frozen="all")


def make_optimizer(name, lr):
    if name == "adam":
        return optimizers.Adam(learning_rate=lr)
    elif name == "sgd":
        return optimizers.SGD(learning_rate=lr, momentum=0.9)
    raise ValueError(name)


def run_hp_experiment(tag, lr, batch_size, epochs, optimizer, dense_units, frozen):
    '''Train one ablation configuration on the HP subset and return a result dict.'''
    model, _ = build_transfer_model(base_name="VGG16", dense_units=dense_units, frozen=frozen)
    model.compile(optimizer=make_optimizer(optimizer, lr),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    hist = model.fit(x_hp, y_hp, validation_split=0.1, batch_size=batch_size,
                      epochs=epochs, verbose=0)
    elapsed = time.time() - t0

    preds = np.argmax(model.predict(x_test, batch_size=128, verbose=0), axis=1)
    acc = accuracy_score(y_true, preds)
    _, _, total = count_params(model)

    result = {
        "Experiment": tag, "Learning Rate": lr, "Batch Size": batch_size,
        "Epochs": epochs, "Optimizer": optimizer, "Dense Units": dense_units,
        "Frozen Layers": frozen, "Test Accuracy": acc,
        "Val Accuracy (last)": hist.history["val_accuracy"][-1],
        "Training Time (s)": round(elapsed, 1), "Total Params": total,
    }
    print(f"[{tag}] test_acc={acc:.4f}  time={elapsed:.1f}s")
    keras.backend.clear_session()
    return result


hp_results = []

In [ ]:
# --- Ablation 1: Learning Rate -> 0.001, 0.0001 ---
for lr in [0.001, 0.0001]:
    cfg = {**HP_BASELINE, "lr": lr}
    hp_results.append(run_hp_experiment(f"LR={lr}", **cfg))

In [ ]:
# --- Ablation 2: Batch Size -> 16, 32, 64 ---
for bs in [16, 32, 64]:
    cfg = {**HP_BASELINE, "batch_size": bs}
    hp_results.append(run_hp_experiment(f"BatchSize={bs}", **cfg))

In [ ]:
# --- Ablation 3: Epochs -> 10, 20 ---
for ep in [10, 20]:
    cfg = {**HP_BASELINE, "epochs": ep}
    hp_results.append(run_hp_experiment(f"Epochs={ep}", **cfg))

In [ ]:
# --- Ablation 4: Optimizer -> Adam, SGD ---
for opt in ["adam", "sgd"]:
    cfg = {**HP_BASELINE, "optimizer": opt}
    hp_results.append(run_hp_experiment(f"Optimizer={opt}", **cfg))

In [ ]:
# --- Ablation 5: Dense Units -> 128, 256 ---
for du in [128, 256]:
    cfg = {**HP_BASELINE, "dense_units": du}
    hp_results.append(run_hp_experiment(f"DenseUnits={du}", **cfg))

In [ ]:
# --- Ablation 6: Frozen Layers -> All, Partial ---
for fr in ["all", "partial"]:
    cfg = {**HP_BASELINE, "frozen": fr}
    hp_results.append(run_hp_experiment(f"Frozen={fr}", **cfg))

In [ ]:
hp_df = pd.DataFrame(hp_results)
hp_df

In [ ]:
# Visual summary of the ablation study: test accuracy per configuration, grouped by factor
factor_groups = {
    "Learning Rate": ["LR=0.001", "LR=0.0001"],
    "Batch Size": ["BatchSize=16", "BatchSize=32", "BatchSize=64"],
    "Epochs": ["Epochs=10", "Epochs=20"],
    "Optimizer": ["Optimizer=adam", "Optimizer=sgd"],
    "Dense Units": ["DenseUnits=128", "DenseUnits=256"],
    "Frozen Layers": ["Frozen=all", "Frozen=partial"],
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (factor, tags) in zip(axes.flat, factor_groups.items()):
    sub = hp_df[hp_df["Experiment"].isin(tags)]
    ax.bar(sub["Experiment"], sub["Test Accuracy"], color="#4c72b0")
    ax.set_title(factor)
    ax.set_ylabel("Test Accuracy")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Hyperparameter Ablation Study — Test Accuracy by Factor", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "06_hyperparameter_ablation")
plt.show()

**TODO AFTER EXECUTION:** Write a 2–3 line data-driven conclusion for each of the six
hyperparameters (which value performed better and a plausible reason), based on the actual
`hp_df` values above — e.g. does a smaller learning rate help or hurt in only 6 epochs, does
batch size trade off speed vs. accuracy, does partial unfreezing beat a fully frozen base even in
this short-run setting, etc.

## 9. Architecture Comparison <a id="arch"></a>

The manual asks for a comparison of **LeNet-5, AlexNet, VGG16, GoogLeNet, and ResNet50** by
parameters, accuracy, and training time. To make this an actual comparison of *architectures*
(rather than repeating the ImageNet-transfer-learning experiment five times), **all five networks
below are trained from scratch on CIFAR-10** — no ImageNet weights are loaded here.

**Why this section switches from Keras to PyTorch/`torchvision`.** `tf.keras.applications` does not
ship AlexNet or GoogLeNet, and hand-written re-implementations risk silently drifting from the
architectures they claim to be (e.g. substituting BatchNorm for AlexNet's original Local Response
Normalization, or dropping GoogLeNet's auxiliary classifiers). To guarantee every model below is the
**actual, authoritative architecture** rather than an approximation, this section uses
`torchvision.models` — the standard reference implementations most published CNN benchmarks are
built on — for **all five** architectures, so the comparison is internally consistent. VGG16 and
ResNet50 were already trained via `keras.applications` earlier in this notebook (Sections 4–8, 11);
here they are rebuilt via `torchvision` purely so all five architectures in this specific comparison
come from the same framework and codebase.

**What "canonical" means for each model here, stated explicitly:**

- **LeNet-5** — implemented directly from LeCun et al. (1998): 2×(5×5 conv → tanh → 2×2 average
  pool) → 3 fully-connected layers (120 → 84 → 10). The **only** change made purely to accommodate
  CIFAR-10 is documented in the code cell below (input channels 1→3); no other architectural
  liberty is taken.
- **AlexNet** — `torchvision.models.alexnet`, i.e. the standard reference AlexNet. `torchvision`'s
  own documentation notes this reproduces the *"One weird trick for parallelizing convolutional
  neural networks"* (Krizhevsky, 2014) single-GPU variant of the original 2012 two-GPU design, which
  is why it has no Local Response Normalization layer (grouped/LRN-based convolutions were an
  engineering artifact of splitting the 2012 network across two GPUs, not part of AlexNet's
  classification logic) — it is **not** a BatchNorm substitution, and no BatchNorm is added anywhere
  in this model.
- **GoogLeNet / Inception-v1** — `torchvision.models.googlenet`, built with `aux_logits=True`, so
  the two **auxiliary classifiers from the original paper are present and used during training**
  (with the paper's 0.3 auxiliary-loss weighting), not omitted as in a simplified reproduction. They
  are automatically bypassed by `torchvision` at inference time, per the original design.
- **VGG16** — `torchvision.models.vgg16`, the standard 16-weight-layer configuration (13 conv + 3
  FC) from Simonyan & Zisserman (2015).
- **ResNet50** — `torchvision.models.resnet50`, the standard 50-layer bottleneck-residual
  architecture from He et al. (2016).

**Adaptations required purely because of CIFAR-10 (RGB, 32×32, 10 classes), documented per model:**

| Model | Adaptation needed for CIFAR-10 | Why |
|---|---|---|
| LeNet-5 | Input channels 1 → 3 | Original LeNet-5 was designed for single-channel (grayscale) MNIST digits; CIFAR-10 is RGB. Output stays 10 classes (digits 0–9 in the original paper, and CIFAR-10's 10 object classes here), so no change is needed there. No resizing is needed: LeNet-5 was designed around ~32×32 input (MNIST digits zero-padded from 28×28 to 32×32), which is exactly CIFAR-10's native resolution. |
| AlexNet, VGG16, GoogLeNet, ResNet50 | Input resized 32×32 → 224×224; final FC/classifier layer set to 10 outputs | These four networks were designed for 224×224 ImageNet input and 1000-way classification. `torchvision`'s constructors accept `num_classes=10` directly (a standard, documented constructor argument, not an architectural edit), so the classifier head is the correct size for CIFAR-10 without touching the convolutional backbone. Resizing 32×32 → 224×224 (bilinear) is done on-the-fly so each backbone receives the input resolution it was designed around; no convolutional/pooling layer is modified. |

**Transparency / limitations of this comparison (stated explicitly, as required):**

- All five models are trained **from randomly-initialized weights** (no ImageNet pretraining) on the
  **same stratified 20% subset** of the CIFAR-10 training data (`x_hp`/`y_hp`, the same subset used
  for the hyperparameter study in Section 8), for the **same 6 epochs**, with the **same batch size
  (32) and Adam optimizer at the same learning rate (0.001)**. This is a deliberately controlled,
  common training budget: it holds data, epochs, batch size, and optimizer constant so that
  differences in the results below are *not* an artifact of one model simply getting more training
  than another.
- **This does not mean the comparison isolates architecture alone.** Four of the five networks
  (AlexNet, VGG16, GoogLeNet, ResNet50) additionally see their input **upsampled to 224×224**, while
  LeNet-5 sees native 32×32 input — an unavoidable consequence of comparing architectures designed
  for very different native resolutions. A short, fixed 6-epoch budget also favours smaller/shallower
  networks that converge quickly over very deep networks (GoogLeNet, ResNet50) that would typically
  need substantially more epochs and/or ImageNet pretraining to reach their well-known high accuracy.
  **The numbers below should be read as short-budget, from-scratch behaviour on this specific setup,
  not as a claim about each architecture's best achievable CIFAR-10 accuracy, nor as a controlled
  ablation of architecture with every other variable held perfectly equal.**
- Parameter counts below will differ slightly from the classic ImageNet-scale figures quoted in the
  theory section and the manual (e.g. AlexNet's ~61M, GoogLeNet's ~6.8M), because those figures
  assume a 1000-class ImageNet output layer; here `num_classes=10` shrinks the final
  classifier/FC layer for every network except LeNet-5 (which used 10 outputs in both cases).


In [ ]:
# PyTorch / torchvision setup for the architecture comparison (Section 9 only).
# All other sections of this notebook use TensorFlow/Keras, per the manual's Task 1 instruction
# to load CIFAR-10 "using TensorFlow/Keras"; PyTorch is scoped to this section so that AlexNet and
# GoogLeNet can be built from their true canonical (torchvision) implementations rather than
# hand-approximated Keras substitutes (see markdown above).
import torch
import torch.nn as nn
import torch.nn.functional as Fun
import torchvision
from torchvision.models.googlenet import GoogLeNetOutputs

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__, "| torchvision version:", torchvision.__version__)
print("Device:", DEVICE)

ARCH_EPOCHS = 6
ARCH_BATCH_SIZE = 32
ARCH_LR = 0.001
RESIZE_TARGET = (224, 224)   # ImageNet-native resolution used by AlexNet/VGG16/GoogLeNet/ResNet50


In [ ]:
class LeNet5(nn.Module):
    '''
    Canonical LeNet-5 (LeCun et al., 1998): 2x(5x5 conv -> tanh -> 2x2 avg-pool),
    followed by 3 fully-connected layers (120 -> 84 -> num_classes).

    The ONLY change from the original architecture, made solely to accommodate CIFAR-10, is the
    number of input channels (3 for RGB, vs. 1 for grayscale MNIST); every layer, activation,
    pooling type, and layer ordering matches the original paper exactly. No resizing is required:
    LeNet-5 was designed around 32x32 input (zero-padded 28x28 MNIST digits), which is exactly
    CIFAR-10's native resolution.
    '''
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(in_channels, 6, kernel_size=5)   # 32x32 -> 28x28
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)            # 14x14 -> 10x10
        self.f5 = nn.Linear(16 * 5 * 5, 120)
        self.f6 = nn.Linear(120, 84)
        self.out = nn.Linear(84, num_classes)

    def forward(self, x):
        x = torch.tanh(self.c1(x))
        x = Fun.avg_pool2d(x, 2)                              # 28x28 -> 14x14
        x = torch.tanh(self.c3(x))
        x = Fun.avg_pool2d(x, 2)                              # 10x10 -> 5x5
        x = torch.flatten(x, 1)
        x = torch.tanh(self.f5(x))
        x = torch.tanh(self.f6(x))
        return self.out(x)


lenet_model_pt = LeNet5(in_channels=3, num_classes=CONFIG["num_classes"])
_dummy = torch.randn(2, 3, 32, 32)
print("LeNet-5 output shape:", lenet_model_pt(_dummy).shape)
print("LeNet-5 parameters:", sum(p.numel() for p in lenet_model_pt.parameters()))


In [ ]:
# Canonical AlexNet, GoogLeNet, VGG16, and ResNet50, all from torchvision.models -- the standard
# reference implementations -- with num_classes set to CIFAR-10's 10 classes and weights=None
# (randomly initialized, trained from scratch below; see markdown above for why).

alexnet_model_pt = torchvision.models.alexnet(weights=None, num_classes=CONFIG["num_classes"])

# aux_logits=True + init_weights=True reproduces the original paper's two auxiliary classifiers,
# used only during training to combat vanishing gradients in this 22-layer network; torchvision
# automatically drops them at inference time (model.eval()), matching the original design.
googlenet_model_pt = torchvision.models.googlenet(
    weights=None, num_classes=CONFIG["num_classes"], aux_logits=True, init_weights=True
)

vgg16_model_pt = torchvision.models.vgg16(weights=None, num_classes=CONFIG["num_classes"])

resnet50_model_pt = torchvision.models.resnet50(weights=None, num_classes=CONFIG["num_classes"])

for _name, _model in [("AlexNet", alexnet_model_pt), ("GoogLeNet", googlenet_model_pt),
                       ("VGG16", vgg16_model_pt), ("ResNet50", resnet50_model_pt)]:
    _n_params = sum(p.numel() for p in _model.parameters())
    print(f"{_name:10s} | torchvision canonical implementation | parameters: {_n_params:,}")


In [ ]:
def train_pytorch_model(model, x, y, epochs=ARCH_EPOCHS, batch_size=ARCH_BATCH_SIZE,
                         lr=ARCH_LR, resize_to=None, has_aux=False, val_split=0.1):
    '''
    Generic training loop shared by all five architectures, so every model gets an identical
    training procedure (same optimizer, LR, batch size, epochs, data split logic).
    '''
    model = model.to(DEVICE)
    n = x.shape[0]
    n_val = int(n * val_split)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    val_idx, train_idx = perm[:n_val], perm[n_val:]
    x_tr, y_tr = x[train_idx], y[train_idx]
    x_val, y_val = x[val_idx], y[val_idx]

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        perm_ep = torch.randperm(x_tr.shape[0])
        running_loss, running_correct = 0.0, 0
        for i in range(0, x_tr.shape[0], batch_size):
            idx = perm_ep[i:i + batch_size]
            xb, yb = x_tr[idx].to(DEVICE), y_tr[idx].to(DEVICE)
            if resize_to is not None:
                xb = Fun.interpolate(xb, size=resize_to, mode="bilinear", align_corners=False)

            optimizer.zero_grad()
            out = model(xb)
            if has_aux and isinstance(out, GoogLeNetOutputs):
                loss = (Fun.cross_entropy(out.logits, yb)
                        + 0.3 * Fun.cross_entropy(out.aux_logits1, yb)
                        + 0.3 * Fun.cross_entropy(out.aux_logits2, yb))
                logits = out.logits
            else:
                loss = Fun.cross_entropy(out, yb)
                logits = out
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(1) == yb).sum().item()

        train_loss = running_loss / x_tr.shape[0]
        train_acc = running_correct / x_tr.shape[0]

        val_loss, val_acc = evaluate_pytorch_model(model, x_val, y_val, batch_size,
                                                     resize_to=resize_to)
        history["loss"].append(train_loss); history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss); history["val_accuracy"].append(val_acc)

    elapsed = time.time() - t0
    return history, elapsed


@torch.no_grad()
def evaluate_pytorch_model(model, x, y, batch_size=128, resize_to=None):
    model.eval()
    total_loss, total_correct = 0.0, 0
    for i in range(0, x.shape[0], batch_size):
        xb, yb = x[i:i + batch_size].to(DEVICE), y[i:i + batch_size].to(DEVICE)
        if resize_to is not None:
            xb = Fun.interpolate(xb, size=resize_to, mode="bilinear", align_corners=False)
        out = model(xb)
        loss = Fun.cross_entropy(out, yb)
        total_loss += loss.item() * xb.size(0)
        total_correct += (out.argmax(1) == yb).sum().item()
    return total_loss / x.shape[0], total_correct / x.shape[0]


@torch.no_grad()
def accuracy_on_test_pytorch(model, x_test_pt, y_test_pt, batch_size=128, resize_to=None):
    _, acc = evaluate_pytorch_model(model, x_test_pt, y_test_pt, batch_size, resize_to=resize_to)
    return acc


In [ ]:
# Reuse the SAME stratified 20% training subset (x_hp / y_hp_raw) built in Section 8 for the
# hyperparameter study, and the SAME held-out test set (x_test / y_true), so this comparison uses
# identical data to the rest of the notebook -- only the modeling framework changes.
x_hp_pt = torch.from_numpy(x_hp).permute(0, 3, 1, 2).float()          # NHWC -> NCHW
y_hp_pt = torch.from_numpy(y_hp_raw.flatten()).long()

x_test_pt = torch.from_numpy(x_test).permute(0, 3, 1, 2).float()
y_test_pt = torch.from_numpy(y_true).long()

print("PyTorch training subset:", x_hp_pt.shape, y_hp_pt.shape)
print("PyTorch test set       :", x_test_pt.shape, y_test_pt.shape)


In [ ]:
arch_models_pt = {
    "LeNet-5":   dict(model=lenet_model_pt,   resize_to=None,          has_aux=False),
    "AlexNet":   dict(model=alexnet_model_pt, resize_to=RESIZE_TARGET, has_aux=False),
    "VGG16":     dict(model=vgg16_model_pt,   resize_to=RESIZE_TARGET, has_aux=False),
    "GoogLeNet": dict(model=googlenet_model_pt, resize_to=RESIZE_TARGET, has_aux=True),
    "ResNet50":  dict(model=resnet50_model_pt, resize_to=RESIZE_TARGET, has_aux=False),
}

arch_results = []
arch_histories = {}

for name, spec in arch_models_pt.items():
    print(f"\n=== Training {name} (torchvision canonical, from scratch) ===")
    model = spec["model"]
    hist, elapsed = train_pytorch_model(
        model, x_hp_pt, y_hp_pt,
        epochs=ARCH_EPOCHS, batch_size=ARCH_BATCH_SIZE, lr=ARCH_LR,
        resize_to=spec["resize_to"], has_aux=spec["has_aux"],
    )
    test_acc = accuracy_on_test_pytorch(model, x_test_pt, y_test_pt, resize_to=spec["resize_to"])
    n_params = sum(p.numel() for p in model.parameters())

    arch_results.append({
        "Architecture": name,
        "Parameters": n_params,
        "Accuracy (%)": round(test_acc * 100, 2),
        "Training Time (s)": round(elapsed, 1),
    })
    arch_histories[name] = hist
    print(f"{name}: params={n_params:,}  test_acc={test_acc:.4f}  time={elapsed:.1f}s")

    # Free memory before the next (potentially large) model trains.
    model.to("cpu")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

arch_df = pd.DataFrame(arch_results)
arch_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].bar(arch_df["Architecture"], arch_df["Parameters"], color="#55a868")
axes[0].set_title("Total Parameters")
axes[0].set_ylabel("Parameters")
axes[0].set_yscale("log")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(arch_df["Architecture"], arch_df["Accuracy (%)"], color="#4c72b0")
axes[1].set_title("Test Accuracy (%)")
axes[1].set_ylabel("Accuracy (%)")
axes[1].tick_params(axis="x", rotation=20)

axes[2].bar(arch_df["Architecture"], arch_df["Training Time (s)"], color="#c44e52")
axes[2].set_title("Training Time (s)")
axes[2].set_ylabel("Seconds")
axes[2].tick_params(axis="x", rotation=20)

for ax in axes:
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Architecture Comparison: LeNet-5 vs. AlexNet vs. VGG16 vs. GoogLeNet vs. ResNet50\n"
             "(torchvision canonical implementations, trained from scratch, common budget)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "07_architecture_comparison")
plt.show()


**TODO AFTER EXECUTION:** Write a 2–3 line data-driven conclusion comparing the five
architectures on parameters vs. accuracy vs. training time from `arch_df` above — noting, for
example, whether GoogLeNet's parameter-efficiency claim holds up under this reduced training
budget, and whether the deepest/most parameter-heavy model was actually the most accurate here.
Remember the stated limitation: all five were trained from scratch on a 20% subset for only 6
epochs (with four of five additionally upsampled to 224×224), so this reflects short-budget,
mixed-resolution behaviour — not each architecture's ceiling performance, and not a comparison that
isolates architecture as the only variable.


## 10. Discussion Questions <a id="discussion"></a>

These are conceptual questions, answerable from theory rather than from this run's specific
numbers, so they are answered directly (not left as TODOs).

**1. Why is AlexNet considered a breakthrough in deep learning?**
It was the first deep CNN to win ImageNet by a large margin (2012), proving deep learning could
outperform hand-engineered computer-vision pipelines. It popularized ReLU activations (faster,
avoids vanishing gradients relative to sigmoid/tanh), Dropout (regularizing a ~61M-parameter
network), and GPU training (making deep networks computationally practical), triggering the
modern deep-learning wave in computer vision.

**2. Why does VGG16 use only 3×3 convolution filters?**
Two stacked 3×3 convolutions have the same receptive field as one 5×5 convolution, but fewer
parameters ($2\times3^2=18$ vs $5^2=25$ per channel) and an extra ReLU non-linearity, giving more
representational capacity per parameter. Using a single, uniform filter size throughout also
makes the architecture simpler to design and reason about.

**3. Explain the advantages of the Inception module.**
It applies multiple filter sizes (1×1, 3×3, 5×5) and pooling in parallel and concatenates them,
capturing features at multiple scales in one layer. 1×1 "bottleneck" convolutions before the
expensive branches drastically cut computation and parameters, which is how GoogLeNet reaches 22
layers with far fewer parameters than the shallower VGG16.

**4. What is the purpose of residual learning?**
It lets a stacked block learn a residual $\mathcal{F}(x)=\mathcal{H}(x)-x$ instead of the full
mapping $\mathcal{H}(x)$ directly. The identity shortcut lets gradients flow straight through the
network during backpropagation, solving the vanishing-gradient/degradation problem that made very
deep plain networks hard to train, and enabling networks with 50-150+ layers.

**5. Differentiate LeNet and ResNet.**
LeNet-5 (1998) is a shallow (7-layer), ~60K-parameter network with average pooling and tanh
activations, designed for simple, low-resolution digit recognition. ResNet (2015) is a very deep
(50+ layer) network using ReLU, batch normalization, and residual/skip connections, designed for
large-scale, high-resolution natural-image classification (ImageNet) — the skip connections are
precisely what make ResNet's much greater depth trainable, something LeNet-5's plain
convolution-pooling stack could never scale to.

**6. What is Transfer Learning?**
Reusing a model already trained on a large source dataset (typically ImageNet) as the starting
point for a new, usually smaller, target task, instead of training from randomly-initialized
weights. The pretrained convolutional filters (general edge/texture/shape detectors) are kept and
only a new task-specific head is trained (or the network is further fine-tuned).

**7. Why is fine tuning required?**
A frozen, ImageNet-pretrained base produces generic features that may not be perfectly suited to
the target domain (e.g. CIFAR-10's low-resolution, differently-distributed images). Fine-tuning —
unfreezing some of the top convolutional layers and continuing training at a small learning rate —
lets the network adapt its higher-level, more task-specific features to the new domain, typically
improving accuracy beyond what a frozen feature extractor alone can achieve.

**8. Explain the difference between dilated convolution and transpose convolution.**
Dilated (atrous) convolution enlarges the receptive field by inserting spacing between kernel
elements, without changing spatial resolution or adding parameters — used to see more context
(e.g. in segmentation) while keeping the feature map size the same. Transpose convolution does the
opposite job: it *increases* spatial resolution (learnable upsampling), turning a small feature
map into a larger one — used in decoders, generators, and segmentation heads that need to recover
resolution.

**9. Why do pretrained models converge faster?**
Their convolutional filters already encode generally useful visual features (edges, textures,
shapes, parts) learned from a large, diverse dataset. The new task only needs to learn how to
recombine or lightly adjust those features for its own classes, rather than learning useful
filters completely from scratch — so fewer epochs and less data are needed to reach good accuracy.

**10. Compare the computational complexity of LeNet and ResNet.**
LeNet-5 has ~60K parameters, 2 convolutional layers, and runs extremely cheaply even on a CPU.
ResNet50 has ~25.6M parameters and 50 layers, requiring substantially more FLOPs, memory, and
(typically) GPU acceleration to train and run inference in reasonable time — several orders of
magnitude more compute for a correspondingly much higher representational capacity and
ImageNet-level accuracy.

## 11. Additional Exercises <a id="exercises"></a>

| # | Exercise | Where it's addressed |
|---|---|---|
| 1 | Implement transfer learning using VGG16 | §4–7 (Tasks 2–5, the core experiment) |
| 2 | Repeat the experiment using ResNet50 | Below (this section) |
| 3 | Compare Adam and SGD optimizers | §8, "Optimizer" ablation |
| 4 | Compare training with frozen layers and fine tuning | §6 (Task 4) and §8, "Frozen Layers" ablation |
| 5 | Evaluate the model using Precision, Recall and F1-score | §7 (Task 5) |
| 6 | Compare the performance of LeNet, AlexNet and ResNet | §9 (`arch_df`, filtered to these three below) |

### Exercise 2 — Repeating the transfer-learning pipeline with ResNet50

In [ ]:
resnet_tl_model, resnet_tl_base = build_transfer_model(
    base_name="ResNet50",
    dense_units=CONFIG["baseline_dense_units"],
    frozen=CONFIG["baseline_frozen"],
)
resnet_tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=CONFIG["baseline_lr"]),
    loss="categorical_crossentropy", metrics=["accuracy"],
)

t0 = time.time()
history_resnet_tl = resnet_tl_model.fit(
    x_train, y_train, validation_split=0.1,
    batch_size=CONFIG["baseline_batch_size"],
    epochs=CONFIG["baseline_epochs"], verbose=1,
)
resnet_tl_time = time.time() - t0

resnet_tl_acc = evaluate_on_test(resnet_tl_model)
_, _, resnet_tl_params = count_params(resnet_tl_model)
print(f"\nResNet50 transfer-learning test accuracy: {resnet_tl_acc:.4f}  "
      f"| params: {resnet_tl_params:,}  | time: {resnet_tl_time:.1f}s")

comparison_vgg_resnet = pd.DataFrame([
    {"Base Model": "VGG16", "Test Accuracy": test_accuracy, "Total Params": total_p,
     "Training Time (s)": round(baseline_training_time + finetune_training_time, 1)},
    {"Base Model": "ResNet50", "Test Accuracy": resnet_tl_acc, "Total Params": resnet_tl_params,
     "Training Time (s)": round(resnet_tl_time, 1)},
])
comparison_vgg_resnet

**TODO AFTER EXECUTION:** Write a 2–3 line comparison of VGG16 vs. ResNet50 as transfer-learning
base models on this task (accuracy, parameter count, training time trade-offs).

### Exercise 6 — LeNet vs. AlexNet vs. ResNet (from `arch_df`, §9)

In [ ]:
arch_df[arch_df["Architecture"].isin(["LeNet-5", "AlexNet", "ResNet50"])]

**TODO AFTER EXECUTION:** Write a 2–3 line comparison of LeNet, AlexNet, and ResNet's
accuracy/parameter/time trade-offs based on the filtered table above.

## 12. Conclusion & References <a id="conclusion"></a>

### Summary of what this notebook accomplished
- Loaded and prepared CIFAR-10 exactly as specified (normalization, sample visualization, shape
  verification).
- Built a VGG16-based transfer-learning pipeline following the manual's exact 6-step workflow,
  trained it with the specified optimizer/LR/batch size/epoch range, then fine-tuned the last
  convolutional block at a reduced learning rate.
- Evaluated the fine-tuned model with the full required metric suite (accuracy, precision,
  recall, F1, confusion matrix, classification report) and visualized results.
- Ran a six-factor hyperparameter ablation study covering every value specified in the manual.
- Implemented and compared five landmark CNN architectures (LeNet-5, AlexNet, VGG16, GoogLeNet,
  ResNet50) from their canonical `torchvision` implementations, trained from scratch under a
  common, documented training budget — explicitly not claimed as a fully isolated architecture-only
  ablation, since input resolution differs by design between LeNet-5 and the other four models.
- Answered all 10 discussion questions and mapped all 6 additional exercises to where they are
  satisfied in the notebook.

**TODO AFTER EXECUTION:** Once the notebook has been run end-to-end, replace this paragraph with
a short (4–6 sentence) overall finding: how much fine-tuning improved accuracy over the frozen
baseline, which hyperparameter choices mattered most, and how the five compared architectures
ranked on the parameters/accuracy/time trade-off under this notebook's training budget.

### References
1. Y. LeCun, L. Bottou, Y. Bengio, P. Haffner, *Gradient-Based Learning Applied to Document
   Recognition*, Proceedings of the IEEE, 1998.
2. A. Krizhevsky, I. Sutskever, G. Hinton, *ImageNet Classification with Deep Convolutional
   Neural Networks*, NeurIPS, 2012.
3. K. Simonyan, A. Zisserman, *Very Deep Convolutional Networks for Large-Scale Image
   Recognition*, ICLR, 2015.
4. C. Szegedy et al., *Going Deeper with Convolutions*, CVPR, 2015.
5. K. He, X. Zhang, S. Ren, J. Sun, *Deep Residual Learning for Image Recognition*, CVPR, 2016.
6. I. Goodfellow, Y. Bengio, A. Courville, *Deep Learning*, MIT Press, 2016.
7. TensorFlow Documentation — https://www.tensorflow.org
8. Keras Documentation — https://keras.io
9. A. Krizhevsky, *Learning Multiple Layers of Features from Tiny Images* (CIFAR-10 dataset),
   2009.
10. PyTorch / `torchvision` Documentation (canonical AlexNet/GoogLeNet/VGG16/ResNet50
    implementations used in Section 9) — https://pytorch.org, https://pytorch.org/vision/
